# 1. Long files

Turns the country questionnaires into one long ("tidy") file per chapter, per
language. **Nothing is translated here** - that is notebook 2.

```
DATA COLLECTOR\datacollector_received_quest_AR\<Chapter>\*.xlsx
        -> COMPENDIUM-ARAB SOCIETY\merged longfiles_AR\<Chapter>_AR.xlsx

DATA COLLECTOR\datacollector_received_quest_EN\<Chapter>\*.xlsx
        -> COMPENDIUM-ARAB SOCIETY\merged longfiles_EN\<Chapter>_EN.xlsx
```

The language is **read off the folder name** - the suffix after
`datacollector_received_quest_` - and that same suffix is used for the output
file name and folder. Nothing lists the languages, so adding a third folder
would just work.

Most questionnaires arrive in Arabic; the English folder is often empty or
partial, and a chapter with no files there simply produces no English long file.

Run this notebook first, then `Compendium_2_Translation.ipynb`.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config / paths


In [ ]:
"""
CELL: Configuration - paths, chapters, the fuzzy-match cutoff, and the column
names the pipeline creates for itself.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
COMPENDIUM_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

# Questionnaire folders are named <prefix><LANGUAGE>. The suffix IS the language,
# so nothing here has to list them - add a folder and it is picked up.
QUESTIONNAIRE_PREFIX = "datacollector_received_quest_"

# One long file per chapter per language lands here.
def long_file_folder(language):
    return COMPENDIUM_PATH / f"merged longfiles_{language}"

# Leave CHAPTERS as None to process every chapter found on disk. Set an explicit
# list to restrict one run, e.g. CHAPTERS = ["Poverty"].
CHAPTERS = None

LANGUAGES = ["AR", "EN"]

# Only used for a sheet whose column names are in neither script.
DEFAULT_LANGUAGE = "AR"

# A fuzzy match must score at least this well (0 to 1) to be used. Below it the
# value is left alone and a warning logged, rather than guessed at.
FUZZY_MATCH_CUTOFF = 0.6

# Column names the pipeline creates itself, written here in Arabic.
YEAR_COLUMN = "\u0627\u0644\u0633\u0646\u0629"
VALUE_COLUMN = "\u0627\u0644\u0639\u062f\u062f"
CHAPTER_COLUMN = "\u0627\u0644\u0641\u0635\u0644"

# Columns used to attach the right source/citation row to each data row:
# year, indicator, country.
MERGE_COLUMNS = ["\u0627\u0644\u0633\u0646\u0629", "\u0627\u0644\u0645\u0624\u0634\u0631", "\u0627\u0644\u062f\u0648\u0644\u0629"]

# Translated normally, but never fuzzy-matched: two citations differing by one
# digit score high enough to overwrite each other.
COLUMNS_NOT_FUZZY_MATCHED = ["\u0627\u0644\u0645\u0635\u062f\u0631"]


## Load the translation dictionary

One dictionary, Arabic to English. It is used here only for the *vocabulary*
that `correct_with_dictionary()` matches misspellings against - nothing is
translated in this notebook.


In [ ]:
"""
CELL: Load translation dict.xlsx - one dictionary, Arabic to English.
"""


def load_dictionary():
    """Reads translation dict.xlsx into:

      DICTIONARY_AR_TO_EN  (column_map, value_map)
          column_map = {Arabic column name: English column name}
          value_map  = {Arabic column name: {Arabic value: English value}}

      ENGLISH_VOCABULARY  (column_names, values_by_column)
          the English column names and values the file knows about.

    Only the Arabic-to-English direction is built, because the questionnaires
    arrive in Arabic and English is what we translate into.

    ENGLISH_VOCABULARY is not a reverse dictionary - it maps nothing. It is just
    the list of correct English spellings, so an English questionnaire's
    misspelled labels can be fuzzy-matched against the right words too.
    """
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            arabic: english
            for arabic, english in zip(rows["val_ar"], rows["val_en"])
            if pd.notna(arabic)
        }

    english_columns = {}
    english_values = {}
    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        english_columns[english_column] = english_column
        english_values[english_column] = {
            str(v): str(v) for v in rows["val_en"].dropna().unique()
        }

    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return (column_map, value_map), (english_columns, english_values), chapter_to_arabic


DICTIONARY_AR_TO_EN, ENGLISH_VOCABULARY, CHAPTER_TO_ARABIC = load_dictionary()


def vocabulary(language):
    """The known column names and values for a language, as
    (column_names, values_by_column) - what misspellings are matched against."""
    return DICTIONARY_AR_TO_EN if language == "AR" else ENGLISH_VOCABULARY


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_name if language == "AR" else arabic_to_english[arabic_name]


arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(f"Dictionary loaded: {len(arabic_columns)} column names, Arabic -> English")


## `questionnaire_folders()`

Finds the input folders and reads each one's language off its name.


In [ ]:
"""
CELL: questionnaire_folders() - find the input folders and read their language.
"""


def questionnaire_folders():
    """Every questionnaire folder present, paired with its language.

    The language is the suffix on the folder name, so it is never configured
    twice: datacollector_received_quest_AR is Arabic, _EN is English, and the
    same suffix names the long file this produces. A folder whose suffix is not
    a language we know is skipped with a warning rather than guessed at.
    """
    found = []
    for folder in sorted(DATA_COLLECTOR_PATH.glob(f"{QUESTIONNAIRE_PREFIX}*")):
        if not folder.is_dir():
            continue
        language = folder.name[len(QUESTIONNAIRE_PREFIX):].strip().upper()
        if language not in LANGUAGES:
            logger.warning(f"Skipping {folder.name}: '{language}' is not one of {LANGUAGES}")
            continue
        found.append((language, folder))

    if not found:
        logger.warning(f"No folders matching {QUESTIONNAIRE_PREFIX}* in {DATA_COLLECTOR_PATH}")
    return found


def discover_chapters():
    """Chapter names taken from the folders on disk rather than a typed list.

    A chapter counts if any language folder has a subfolder of that name holding
    at least one .xlsx - so a new chapter needs no configuration, and an empty
    folder does not produce an empty output file.
    """
    names = set()
    for _, questionnaire_root in questionnaire_folders():
        for child in questionnaire_root.iterdir():
            if child.is_dir() and any(
                f for f in child.glob("*.xlsx") if not f.name.startswith("~$")
            ):
                names.add(child.name)
    return sorted(names)


def chapters_to_process():
    """CHAPTERS if it was set, otherwise whatever is on disk."""
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters discovered on disk: {found}")
    return found


## `extract_tables()`

Each sheet holds two tables marked by an `index` column: `index=1` is the data,
`index=2` is the source. The row reading `index` in column 0 carries the column
names for the table below it.


In [ ]:
"""
CELL: extract_tables() - split one raw sheet into its data table and source table.
"""


def extract_tables(raw_sheet):
    header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
    data_header_row, source_header_row = header_rows[0], header_rows[1]

    # .str.strip() matters here: a stray trailing space in a raw header (seen
    # in a couple of Morocco files) would otherwise silently break the merge
    # key match in reshape_and_merge() and cause a duplicate-column crash in
    # correct_with_dictionary().
    data_columns = raw_sheet.iloc[data_header_row].dropna().str.strip()
    data_table = raw_sheet[raw_sheet[0] == "1"][data_columns.index].copy()
    data_table.columns = data_columns.values
    data_table = data_table.drop(columns=["index"])

    source_columns = raw_sheet.iloc[source_header_row].dropna().str.strip()
    source_table = raw_sheet[raw_sheet[0] == "2"][source_columns.index].copy()
    source_table.columns = source_columns.values
    source_table = source_table.drop(columns=["index"])

    return data_table, source_table


## `detect_language()`

Works out whether a sheet is written in Arabic or English **by looking at the
script its column names are written in** - Arabic letters occupy their own
Unicode block, Latin letters another. No dictionary needed, so it still works
on a column the dictionary has never seen.

The pipeline does not use this to decide what to do - `TRANSLATE_TO` and the
folder do that. It is used as a check: if a sheet in the Arabic folder turns
out to be English, you get a warning naming the file and sheet instead of a
silent mess of untranslated rows.


In [ ]:
"""
CELL: detect_language() - is this sheet written in Arabic or English?
"""


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it."""
    return any("\u0600" <= character <= "\u06ff" for character in str(text))


def detect_language(table):
    """Returns "AR" or "EN" based on the script the column names are written in.

    Column names are a far better signal than cell values: there are only a
    couple of dozen of them, while values include numbers, codes and
    country-specific free text that can be in either script.

    Falls back to DEFAULT_LANGUAGE if the table has no usable column names.
    """
    names = [str(c) for c in table.columns if not str(c).isdigit()]
    if not names:
        return DEFAULT_LANGUAGE

    arabic_names = sum(1 for name in names if looks_arabic(name))
    return "AR" if arabic_names > len(names) / 2 else "EN"


## `reshape_and_merge()`

Unpivots the data table's year columns (any column whose name is all
digits) into two columns (year, value), then merges in the matching row
from the source table so every data point carries its source.

Takes the sheet's `language` so the two columns it creates, and the columns
it merges on, are named in that same language.


In [ ]:
"""
CELL: reshape_and_merge() - wide-to-long reshape, then attach the source table.
"""


def reshape_and_merge(data_table, source_table, language):
    id_columns = [c for c in data_table.columns if not str(c).isdigit()]
    year_columns = [c for c in data_table.columns if str(c).isdigit()]

    long_table = data_table.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name=column_name(YEAR_COLUMN, language),
        value_name=column_name(VALUE_COLUMN, language),
    )

    # Merge on year/indicator/country spelled in the sheet's own language,
    # and only on the ones both tables actually have.
    merge_columns = [column_name(c, language) for c in MERGE_COLUMNS]
    merge_columns = [c for c in merge_columns if c in long_table.columns and c in source_table.columns]

    # Trim whitespace on BOTH sides of the merge keys first. This step runs
    # before correct_with_dictionary(), so a key with a stray trailing space
    # matches nothing in the source table and the row loses its citation
    # silently - the spelling is corrected a step later, too late to help. It is
    # not a rare case: of one Labor run's 358 fuzzy corrections, 331 were pure
    # whitespace, and one Algeria sheet lost ten citations to a single trailing
    # space on an indicator name.
    long_table = long_table.copy()
    source_table = source_table.copy()
    for column in merge_columns:
        long_table[column] = long_table[column].astype(str).str.replace("\xa0", " ").str.strip()
        source_table[column] = source_table[column].astype(str).str.replace("\xa0", " ").str.strip()

    return pd.merge(long_table, source_table, on=merge_columns, how="left")


## `correct_with_dictionary()`

Works in whichever language the sheet is written in - the `language` argument
picks which vocabulary to match against, so an English sheet is checked
against the English column names and values, an Arabic one against the Arabic.

For every column: if its name is already a known column name in that language,
leave it. Otherwise, find the known column name it's most similar to, and
rename it there - but only if that similarity score clears
`FUZZY_MATCH_CUTOFF`.

Then do the same thing for every value within each known column: if the
value is already known, leave it; otherwise replace it with the closest
known value, if close enough. Every actual change is printed.

Values in the **Source** column are never fuzzy-matched (see
`COLUMNS_NOT_FUZZY_MATCHED`) - two citations differing by a single digit
score high enough to overwrite each other, so the nearest entry would be a
wrong answer rather than a correction. They are still *translated* by exact
lookup. A misspelled Source *header* is still fixed, which is what folds
Iraq's `المصادر` back into the Source column.


In [ ]:
"""
CELL: correct_with_dictionary() - fix column names and cell values, in either language.
"""


def best_match(text, choices):
    """Compares text against every choice and returns (best_choice, score) -
    the one difflib considers most similar, and how similar (0 to 1)."""
    best_choice, best_score = None, -1
    for choice in choices:
        score = difflib.SequenceMatcher(None, str(text), str(choice)).ratio()
        if score > best_score:
            best_choice, best_score = choice, score
    return best_choice, best_score


def correct_with_dictionary(table, language, chapter, file_name, sheet_name):
    """Fixes misspelled column names and values by matching them against the
    dictionary's vocabulary for `language` ("AR" or "EN")."""
    known_columns, known_values_by_column = vocabulary(language)
    never_guessed = [column_name(c, language) for c in COLUMNS_NOT_FUZZY_MATCHED]

    table = table.copy()
    replacements = 0
    where = f"[{chapter}/{language}] {file_name} | {sheet_name}"

    # 1. Fix column names.
    for column in list(table.columns):
        if column in known_columns:
            continue  # already a known column name, nothing to fix

        match, score = best_match(column, known_columns.keys())
        if match is not None and score >= FUZZY_MATCH_CUTOFF:
            print(f"{where} | column: {column} -> {match} (score={score:.2f})")
            table = table.rename(columns={column: match})
            replacements += 1
        elif match is not None:
            logger.warning(
                f"{where} | column '{column}' has no good match "
                f"(closest is '{match}', score={score:.2f}) - left unchanged"
            )

    # 2. Fix cell values, one known column at a time.
    for column in table.columns:
        if column in never_guessed:
            continue  # free-text citations - translated, but never guessed at

        known_values = known_values_by_column.get(column)
        if not known_values:
            continue  # not a dictionary column, or it has no fixed vocabulary (e.g. Year, Value)

        for value in table[column].dropna().unique():
            if value in known_values:
                continue  # already a known value, nothing to fix

            match, score = best_match(value, known_values.keys())
            if match is not None and score >= FUZZY_MATCH_CUTOFF:
                print(f"{where} | {column}: {value} -> {match} (score={score:.2f})")
                table[column] = table[column].replace(value, match)
                replacements += 1
            elif match is not None:
                logger.warning(
                    f"{where} | {column} value '{value}' has no good match "
                    f"(closest is '{match}', score={score:.2f}) - left unchanged"
                )

    return table, replacements


## Error tracking

`log_failure()` is the one place that logs a failure and remembers it, so
the final cell can print one consolidated summary grouped by chapter.

In [ ]:
"""
CELL: Error tracking - log a failure and remember it for the run summary.
"""
FAILURES = defaultdict(list)  # chapter -> list of {file, sheet, step, error}

# Plain-language guess at what's wrong, keyed by which step failed.
LIKELY_CAUSES = {
    "read": "file may be corrupted, password-protected, or not a valid .xlsx",
    "extract": "sheet may be missing an 'index' column, or index values are not 1/2 as expected",
    "detect language": "sheet's column names match neither the Arabic nor the English dictionary",
    "reshape_and_merge": "data table may be missing year columns, or the merge columns don't match the source table",
    "dictionary correction": "column names or values may be malformed and unable to be matched",
    "translation": "a column or value may not have a corresponding entry in the other language",
}


def log_failure(chapter, file_name, sheet_name, step, error):
    likely_cause = LIKELY_CAUSES.get(step, "unexpected error, inspect the sheet manually")
    logger.error(
        f"[ERROR] Chapter={chapter} | File={file_name} | Sheet={sheet_name} | "
        f"Step={step} | {type(error).__name__}: {error} | Likely cause: {likely_cause}"
    )
    FAILURES[chapter].append({
        "file": file_name,
        "sheet": sheet_name,
        "step": step,
        "error": f"{type(error).__name__}: {error}",
    })


## `process_chapter()`

Every step, for every sheet, for every file in one chapter of one language
folder. Each sheet runs inside one try/except with a `step` variable tracking
the stage, so a failure is logged against the right step without needing a
try/except per step.


In [ ]:
"""
CELL: process_chapter() - every step for one chapter of one language folder.
"""


def process_chapter(chapter, language, questionnaire_root):
    """Builds one long file from one chapter folder, and returns how many rows
    it wrote (0 if there was nothing to read)."""
    folder = questionnaire_root / chapter
    if not folder.exists():
        logger.info(f"  {language}/{chapter}: no folder, skipping")
        return 0

    files = sorted(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$"))
    if not files:
        logger.info(f"  {language}/{chapter}: folder is empty, skipping")
        return 0

    logger.info(f"  {language}/{chapter}: {len(files)} file(s)")

    tables = []
    sheets_done = 0
    wrong_language = 0
    replacements_made = 0

    for file_path in files:
        file_name = file_path.name
        try:
            xls = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as error:
            log_failure(chapter, file_name, "-", "read", error)
            continue

        for sheet_name in xls.sheet_names:
            step = "read"
            try:
                raw_sheet = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)

                step = "extract"
                data_table, source_table = extract_tables(raw_sheet)

                step = "detect language"
                # A check only - the FOLDER decides. A mismatch means a file has
                # been filed in the wrong folder, which would otherwise surface
                # as a sheet of labels the dictionary cannot match.
                if detect_language(data_table) != language:
                    wrong_language += 1
                    logger.warning(
                        f"    {file_name} | {sheet_name}: looks like "
                        f"{detect_language(data_table)} but sits in the {language} "
                        f"folder - processing it as {language}"
                    )

                chapter_in_language = chapter if language == "EN" else CHAPTER_TO_ARABIC[chapter]
                data_table[column_name(CHAPTER_COLUMN, language)] = chapter_in_language

                step = "reshape_and_merge"
                merged_table = reshape_and_merge(data_table, source_table, language)

                step = "dictionary correction"
                corrected_table, n = correct_with_dictionary(
                    merged_table, language, chapter, file_name, sheet_name
                )
                replacements_made += n

            except Exception as error:
                log_failure(chapter, file_name, sheet_name, step, error)
                continue

            tables.append(corrected_table)
            sheets_done += 1

    if not sheets_done:
        logger.warning(f"  {language}/{chapter}: nothing extracted, no file written")
        return 0

    result = pd.concat(tables, ignore_index=True)
    folder_out = long_file_folder(language)
    folder_out.mkdir(parents=True, exist_ok=True)
    path = folder_out / f"{chapter}_{language}.xlsx"
    result.to_excel(path, index=False, engine="openpyxl")

    logger.info(
        f"  {language}/{chapter}: {sheets_done} sheet(s), {replacements_made} correction(s)"
        + (f", {wrong_language} sheet(s) looked like the wrong language" if wrong_language else "")
        + f" -> {folder_out.name}\\{path.name} ({len(result):,} rows)"
    )
    return len(result)


## Run - build every long file

Walks each questionnaire folder in turn, and each chapter inside it. A chapter
with no folder or no files is skipped and noted, so an empty English folder is
normal rather than an error.


In [ ]:
"""
CELL: Main run - build a long file for every chapter of every language folder.
"""
FAILURES.clear()

folders = questionnaire_folders()
print(f"Questionnaire folders found: {[f'{lang} ({f.name})' for lang, f in folders]}")
print(f"Writing long files to: {COMPENDIUM_PATH}\\merged longfiles_<LANG>\n")

written = {}
total_steps = len(folders) * len(CHAPTERS)
step_number = 0

for language, questionnaire_root in folders:
    print(f"\n=== {language}  ({questionnaire_root.name}) ===")
    for chapter in chapters_to_process():
        step_number += 1
        bar = "#" * step_number + "-" * (total_steps - step_number)
        print(f"[{bar}] {step_number}/{total_steps}  {language}/{chapter}")
        rows = process_chapter(chapter, language, questionnaire_root)
        if rows:
            written[(language, chapter)] = rows

print("\n" + "=" * 70)
print("LONG FILES WRITTEN")
print("=" * 70)
if not written:
    print("None - check the questionnaire folders.")
else:
    for (language, chapter), rows in sorted(written.items()):
        print(f"  merged longfiles_{language}\\{chapter}_{language}.xlsx   {rows:>9,} rows")
    for language in LANGUAGES:
        chapters = [c for (l, c) in written if l == language]
        print(f"\n  {language}: {len(chapters)} chapter(s) - {sorted(chapters) if chapters else 'none'}")


## Run summary - failures grouped by chapter

In [ ]:
"""
CELL: Run summary - every failure from the run above, grouped by chapter.
"""
print("\n" + "=" * 70)
print("RUN SUMMARY - FAILURES BY CHAPTER")
print("=" * 70)

if not FAILURES:
    print("No failures. All files/sheets processed successfully.")
else:
    total = sum(len(v) for v in FAILURES.values())
    print(f"{total} failure(s) across {len(FAILURES)} chapter(s):\n")
    for chapter, failures in FAILURES.items():
        print(f"{chapter} ({len(failures)} failure(s)):")
        for f in failures:
            print(f"  - {f['file']} | Sheet={f['sheet']} | Step={f['step']} | {f['error']}")
        print()
